In [ ]:
import re
import time
import pandas as pd

from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.common.keys import Keys
from selenium.webdriver.common.action_chains import ActionChains
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC


APPLY_URL = "https://www.gov.kr/mw/AA040OfferMainFrm.do?capp_biz_cd=13100000026&HighCtgCD=-&FAX_TYPE=B&img=02&selectedSeq=01"

INPUT_FILE = "병합데이터.xlsx"      # 소재지 컬럼 포함
OUTPUT_FILE = "gov24_result.xlsx"

WAIT = 20
TEST_N = None   # 테스트 시 숫자 입력


# =========================
# 1. 공통 함수
# =========================
def clean_text(x):
    if pd.isna(x):
        return ""
    return " ".join(str(x).strip().split())


def normalize_address(addr):
    if pd.isna(addr):
        return None

    addr = clean_text(addr)
    if addr == "":
        return None

    addr = re.sub(r"\s+", " ", addr)
    addr = re.sub(r"\s*-\s*", "-", addr)

    addr = addr.replace("울산시", "울산광역시")
    if addr.startswith("울산 "):
        addr = addr.replace("울산 ", "울산광역시 ", 1)

    if not addr.startswith("울산광역시"):
        addr = "울산광역시 " + addr

    return addr


def load_addresses():
    df = pd.read_excel(INPUT_FILE)
    df.columns = [clean_text(c) for c in df.columns]

    if "소재지" not in df.columns:
        raise KeyError("입력 파일에 '소재지' 컬럼이 없습니다.")

    df = df.copy()
    df["소재지"] = df["소재지"].apply(normalize_address)
    df = df[df["소재지"].notna()].reset_index(drop=True)

    if TEST_N:
        df = df.iloc[:TEST_N].copy()

    return df


# =========================
# 2. 주소 파싱
# =========================
def parse_address_for_gov(addr):
    """
    예:
    울산광역시 남구 매암동 360-33
    울산광역시 울주군 온산읍 산암리 143-1
    """
    addr = normalize_address(addr)
    if not addr:
        return None

    parts = addr.split()

    if len(parts) == 4:
        sido, sigungu, dong_or_eup, lot = parts
        ri = None
    elif len(parts) == 5:
        sido, sigungu, dong_or_eup, ri, lot = parts
    else:
        return None

    lot = lot.strip()
    is_san = False
    if lot.startswith("산"):
        is_san = True
        lot = lot.replace("산", "", 1).strip()

    if "-" in lot:
        bonbun, bubun = lot.split("-", 1)
    else:
        bonbun, bubun = lot, ""

    bonbun = bonbun.strip()
    bubun = bubun.strip()

    if not bonbun:
        return None

    # 정부24 검색용 주소
    # 리가 없으면 "울산광역시 남구 매암동"
    # 리가 있으면 "울산광역시 울주군 온산읍 산암리"
    if ri:
        dong_search = f"{sido} {sigungu} {dong_or_eup} {ri}"
    else:
        dong_search = f"{sido} {sigungu} {dong_or_eup}"

    return {
        "원주소": addr,
        "sido": sido,
        "sigungu": sigungu,
        "dong_or_eup": dong_or_eup,
        "ri": ri,
        "bonbun": bonbun,
        "bubun": bubun,
        "is_san": is_san,
        "dong_search": dong_search,
    }


# =========================
# 3. 정부24 특수 매핑
# =========================
def build_dong_select(info):
    """
    정부24 팝업에서 선택해야 하는 표시명 생성.
    기본은 검색어와 동일하게 찾고,
    매암동처럼 행정동 통합 표기가 필요한 경우 예외 매핑.
    """
    search = info["dong_search"]

    special_map = {
        "울산광역시 남구 매암동": "울산광역시 남구 야음장생포동(매암동)",
    }

    return special_map.get(search, search)


# =========================
# 4. 클릭 유틸
# =========================
def click_strong(driver, elem):
    methods = [
        lambda: elem.click(),
        lambda: driver.execute_script("arguments[0].click();", elem),
        lambda: driver.execute_script("""
            const e = arguments[0];
            e.dispatchEvent(new MouseEvent('mousedown', {bubbles:true}));
            e.dispatchEvent(new MouseEvent('mouseup', {bubbles:true}));
            e.dispatchEvent(new MouseEvent('click', {bubbles:true}));
        """, elem),
        lambda: ActionChains(driver).move_to_element(elem).pause(0.2).click().perform(),
    ]
    for method in methods:
        try:
            method()
            return True
        except:
            pass
    return False


# =========================
# 5. 팝업 처리
# =========================
def close_survey_popup(driver):
    xpaths = [
        "//button[normalize-space()='다음에 하기']",
        "//*[normalize-space()='다음에 하기']",
        "//button[contains(., '다음에 하기')]",
        "//*[contains(normalize-space(.), '다음에 하기')]",
    ]
    for xp in xpaths:
        try:
            elem = WebDriverWait(driver, 1).until(
                EC.presence_of_element_located((By.XPATH, xp))
            )
            if click_strong(driver, elem):
                return True
        except:
            pass

    try:
        ActionChains(driver).send_keys(Keys.ESCAPE).perform()
    except:
        pass
    return False


def close_alert_if_present(driver):
    try:
        alert = driver.switch_to.alert
        text = alert.text
        alert.accept()
        return text
    except:
        return None


# =========================
# 6. 주소 선택
# =========================
def select_address_result(driver, wait, dong_select):
    candidates = [
        f"//dd[.//div[contains(normalize-space(.), '{dong_select}')]]",
        f"//*[contains(normalize-space(.), '{dong_select}')]",
    ]

    last_error = None
    for xp in candidates:
        try:
            result = wait.until(EC.presence_of_element_located((By.XPATH, xp)))
            if click_strong(driver, result):
                return
        except Exception as e:
            last_error = e

    raise Exception(f"주소 선택 실패: {dong_select}") from last_error


def go_apply_page(driver):
    driver.get(APPLY_URL)


# =========================
# 7. 한 건 처리
# =========================
def process_one(driver, wait, address):
    info = parse_address_for_gov(address)
    if info is None:
        return {
            "정부24상태": "주소파싱실패",
            "정부24메모": "",
        }

    dong_search = info["dong_search"]
    dong_select = build_dong_select(info)
    bonbun = info["bonbun"]
    bubun = info["bubun"]

    main_window = driver.current_window_handle
    before_windows = driver.window_handles[:]

    # 주소검색 버튼
    addr_btn = wait.until(EC.presence_of_element_located((By.ID, "btnAddress")))
    if not click_strong(driver, addr_btn):
        raise Exception("주소검색 버튼 클릭 실패")

    # 팝업 전환
    wait.until(lambda d: len(d.window_handles) > len(before_windows))
    popup = [w for w in driver.window_handles if w not in before_windows][0]
    driver.switch_to.window(popup)

    # 주소 검색
    dong_input = wait.until(EC.element_to_be_clickable((By.ID, "txtAddr")))
    dong_input.clear()
    dong_input.send_keys(dong_search)
    dong_input.send_keys(Keys.ENTER)

    # 주소 선택
    select_address_result(driver, wait, dong_select)

    # 본창 복귀
    wait.until(lambda d: len(d.window_handles) == 1)
    driver.switch_to.window(main_window)

    # 본번
    bonbun_input = wait.until(EC.element_to_be_clickable((
        By.ID,
        "토지임야대장신청서_IN-토지임야대장신청서_신청토지소재지_주소정보_상세주소_번지"
    )))
    bonbun_input.clear()
    bonbun_input.send_keys(str(bonbun))

    # 부번
    bubun_input = wait.until(EC.element_to_be_clickable((
        By.ID,
        "토지임야대장신청서_IN-토지임야대장신청서_신청토지소재지_주소정보_상세주소_호"
    )))
    bubun_input.clear()
    if bubun:
        bubun_input.send_keys(str(bubun))

    # 산 여부는 사이트 구조 확인 후 필요하면 추가
    # 현재는 네 원본 코드 기준 유지

    # 신청하기
    apply_btn = wait.until(EC.presence_of_element_located((By.ID, "btn_end")))
    if not click_strong(driver, apply_btn):
        raise Exception("신청하기 버튼 클릭 실패")

    time.sleep(0.8)

    alert_text = close_alert_if_present(driver)
    survey_closed = close_survey_popup(driver)

    time.sleep(1.2)

    status = "신청완료대기"
    memo = ""

    if alert_text:
        status = "오류팝업"
        memo = alert_text
    elif survey_closed:
        status = "신청완료"
    else:
        status = "신청완료"

    return {
        "정부24상태": status,
        "정부24메모": memo,
    }


# =========================
# 8. 실행
# =========================
def main():
    df = load_addresses()

    if "정부24상태" not in df.columns:
        df["정부24상태"] = ""
    if "정부24메모" not in df.columns:
        df["정부24메모"] = ""

    options = Options()
    options.add_experimental_option("detach", True)

    driver = webdriver.Chrome(options=options)
    wait = WebDriverWait(driver, WAIT)

    try:
        go_apply_page(driver)
        input("정부24 로그인 후 엔터...")

        for i, row in df.iterrows():
            address = row["소재지"]
            print(f"[{i+1}/{len(df)}] {address}")

            try:
                go_apply_page(driver)
                result = process_one(driver, wait, address)

                df.loc[i, "정부24상태"] = result["정부24상태"]
                df.loc[i, "정부24메모"] = result["정부24메모"]

            except Exception as e:
                print("오류:", e)
                df.loc[i, "정부24상태"] = "확인필요"
                df.loc[i, "정부24메모"] = str(e)

                try:
                    close_alert_if_present(driver)
                except:
                    pass

            time.sleep(0.8)

        df.to_excel(OUTPUT_FILE, index=False)
        print("완료:", OUTPUT_FILE)

    finally:
        try:
            driver.quit()
        except:
            pass


if __name__ == "__main__":
    main()

정부24 로그인 후 엔터... 


[1/393] 울산광역시 남구 고사동 1-2
오류: 주소 선택 실패: 울산광역시 남구 고사동
[2/393] 울산광역시 남구 고사동 110-26
오류: Message: 

[3/393] 울산광역시 남구 고사동 110-27
오류: Message: 

[4/393] 울산광역시 남구 고사동 110-28



KeyboardInterrupt


KeyboardInterrupt

